# 24 — Silver Web Collection Specification

## Objetivo

Preparar la recolección del conjunto `OWN_LLM_SILVER_WEB` que servirá como fuente principal de desarrollo del clasificador.

La unidad metodológica será:

\[
	ext{sitio} + 	ext{página/tarea} + 	ext{ejecución}
\]

y cada instancia tendrá:

- **X**: métricas canónicas capturadas por la extensión;
- **Y**: H1/H4/H5/H6 evaluadas por LLM de forma independiente de las X calculadas;
- **procedencia**: versión de prompt, modelo, ejecución y evidencia usada;
- **agrupamiento**: `site_id`, para impedir fuga entre train y test.

Este notebook **no recopila datos todavía**. Genera contratos, plantillas y criterios de aceptación reproducibles.


## 1. Configuración

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import textwrap

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 400)

PROJECT_DIR = Path("..").resolve()

REPORTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "tables"
)

SILVER_DIR = (
    PROJECT_DIR
    / "data"
    / "silver"
)

SILVER_METADATA_DIR = (
    SILVER_DIR
    / "metadata"
)

SILVER_RAW_DIR = (
    SILVER_DIR
    / "raw"
)

SILVER_LABEL_DIR = (
    SILVER_DIR
    / "labels"
)

SILVER_INTERIM_DIR = (
    SILVER_DIR
    / "interim"
)

for directory in [
    REPORTS_DIR,
    SILVER_METADATA_DIR,
    SILVER_RAW_DIR,
    SILVER_LABEL_DIR,
    SILVER_INTERIM_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("SILVER_DIR:", SILVER_DIR)


SILVER_DIR: /Users/uriel/Documents/machine_learning/data/silver


## 2. Cargar las 40 X canónicas

In [2]:
dictionary_path = (
    REPORTS_DIR
    / "feature_dictionary_validation.csv"
)

dictionary = pd.read_csv(
    dictionary_path
)

canonical_x = (
    dictionary[
        dictionary[
            "rol_ml"
        ]
        ==
        "X"
    ]
    .copy()
)

print(
    "Diccionario:",
    dictionary.shape
)

print(
    "X canónicas:",
    len(
        canonical_x
    )
)

assert len(canonical_x) == 40

display(
    canonical_x
)


Diccionario: (48, 10)
X canónicas: 40


,variable,categoria,rol_ml,a11y_estado,repo01_estado,delfi_estado,irr_aporte,requiere_experto,captura_extension,estado_final
4,task_time_s,TEMPORAL,X,DIRECTA_O_DERIVADA,NO,NO,NO,False,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS
5,mean_response_time_ms,TEMPORAL,X,NO,NO,NO,NO,False,DERIVADA,RECOLECCION_PROPIA
6,max_response_time_ms,TEMPORAL,X,NO,NO,NO,NO,False,DERIVADA,RECOLECCION_PROPIA
7,mean_pre_action_pause_s,TEMPORAL,X,NO,DERIVABLE_REQUIERE_VALIDACION,NO,NO,False,DERIVADA,RESPALDADA_POR_DATOS_PUBLICOS
8,click_count,INTERACCION,X,DIRECTA_O_DERIVADA,DERIVABLE_VALIDADA_EN_MUESTRA,NO,NO,False,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS
9,keypress_count,INTERACCION,X,DIRECTA_O_DERIVADA,VALIDADA,NO,NO,False,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS
10,scroll_count,INTERACCION,X,DIRECTA_O_DERIVADA,DERIVABLE,NO,NO,False,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS
11,backtracking_count,INTERACCION,X,DIRECTA_O_DERIVADA,NO,NO,NO,False,DERIVADA,RESPALDADA_POR_DATOS_PUBLICOS
12,repeated_action_count,INTERACCION,X,DIRECTA_O_DERIVADA,DERIVABLE_CON_REGLA_EXPERIMENTAL,NO,NO,False,DERIVADA,RESPALDADA_POR_DATOS_PUBLICOS
13,navigation_steps,INTERACCION,X,DIRECTA_O_DERIVADA,PARCIAL,NO,NO,False,DERIVADA,RESPALDADA_POR_DATOS_PUBLICOS


## 3. Contrato de captura de X

La extensión exportará las X canónicas sin targets. Esto mantiene separada la medición automática de la anotación Nielsen.


In [3]:
X_NAMES = (
    canonical_x[
        "variable"
    ]
    .astype(str)
    .tolist()
)

capture_contract_rows = [
    {
        "column":
            "instance_id",

        "role":
            "ID",

        "required":
            True,

        "description":
            "Identificador único de la observación."
    },

    {
        "column":
            "site_id",

        "role":
            "GROUP",

        "required":
            True,

        "description":
            "Identificador estable del sitio/aplicación; se usará para grouped split."
    },

    {
        "column":
            "task_id",

        "role":
            "CONTEXT",

        "required":
            True,

        "description":
            "Tarea reproducible ejecutada en la interfaz."
    },

    {
        "column":
            "execution_id",

        "role":
            "CONTEXT",

        "required":
            True,

        "description":
            "Ejecución concreta de la tarea."
    },

    {
        "column":
            "platform",

        "role":
            "CONTEXT",

        "required":
            True,

        "description":
            "Sitio/aplicación evaluada, derivado del dominio o aplicación."
    },

    {
        "column":
            "page_type",

        "role":
            "CONTEXT",

        "required":
            True,

        "description":
            "Tipo de página según regla documentada."
    }
]

for variable in X_NAMES:

    capture_contract_rows.append({
        "column":
            variable,

        "role":
            "X",

        "required":
            False,

        "description":
            "Métrica canónica definida en feature_dictionary_validation.csv. NaN si no es observable/aplicable."
    })

extension_capture_contract = pd.DataFrame(
    capture_contract_rows
)

print(
    "Columnas del contrato:",
    len(
        extension_capture_contract
    )
)

print(
    "X incluidas:",
    (
        extension_capture_contract[
            "role"
        ]
        ==
        "X"
    ).sum()
)

display(
    extension_capture_contract
)


Columnas del contrato: 46
X incluidas: 40


,column,role,required,description
0,instance_id,ID,True,Identificador único de la observación.
1,site_id,GROUP,True,Identificador estable del sitio/aplicación; se usará para grouped split.
2,task_id,CONTEXT,True,Tarea reproducible ejecutada en la interfaz.
3,execution_id,CONTEXT,True,Ejecución concreta de la tarea.
4,platform,CONTEXT,True,"Sitio/aplicación evaluada, derivado del dominio o aplicación."
5,page_type,CONTEXT,True,Tipo de página según regla documentada.
6,task_time_s,X,False,Métrica canónica definida en feature_dictionary_validation.csv. NaN si no es observable/aplicable.
7,mean_response_time_ms,X,False,Métrica canónica definida en feature_dictionary_validation.csv. NaN si no es observable/aplicable.
8,max_response_time_ms,X,False,Métrica canónica definida en feature_dictionary_validation.csv. NaN si no es observable/aplicable.
9,mean_pre_action_pause_s,X,False,Métrica canónica definida en feature_dictionary_validation.csv. NaN si no es observable/aplicable.


### Regla de faltantes

`NaN` significa **no observable, no aplicable o no medido** según la definición de la variable.

No se hará `fillna(0)` global. Cero solo se utiliza cuando la operacionalización de la variable permite afirmar empíricamente ausencia/cantidad cero.


## 4. Unidad de observación

In [4]:
observation_unit_policy = pd.DataFrame([
    {
        "field":
            "site_id",

        "definition":
            "Aplicación/sitio web estable.",

        "example":
            "site_001"
    },

    {
        "field":
            "task_id",

        "definition":
            "Tarea predefinida con criterio de inicio y éxito.",

        "example":
            "task_login"
    },

    {
        "field":
            "execution_id",

        "definition":
            "Ejecución concreta de una tarea en un sitio.",

        "example":
            "exec_001"
    },

    {
        "field":
            "instance_id",

        "definition":
            "site_id + task_id + execution_id, codificado de forma única.",

        "example":
            "site_001__task_login__exec_001"
    }
])

display(
    observation_unit_policy
)


,field,definition,example
0,site_id,Aplicación/sitio web estable.,site_001
1,task_id,Tarea predefinida con criterio de inicio y éxito.,task_login
2,execution_id,Ejecución concreta de una tarea en un sitio.,exec_001
3,instance_id,"site_id + task_id + execution_id, codificado de forma única.",site_001__task_login__exec_001


## 5. Registro de sitios

In [5]:
site_registry_columns = [
    "site_id",
    "site_name",
    "base_url",
    "domain",
    "educational_context",
    "access_type",
    "requires_account",
    "uses_personal_data",
    "site_version_or_date",
    "included",
    "exclusion_reason",
    "notes"
]

site_registry_template = pd.DataFrame(
    columns=site_registry_columns
)

display(
    site_registry_template
)


,site_id,site_name,base_url,domain,educational_context,access_type,requires_account,uses_personal_data,site_version_or_date,included,exclusion_reason,notes


### Criterios de inclusión

Para el piloto, priorizar interfaces educativas web que puedan evaluarse de forma reproducible y ética:

- acceso público, demo o cuenta de prueba autorizada;
- tareas concretas y repetibles;
- sin introducir datos personales reales;
- suficiente interacción para observar al menos parte de H1/H4/H5/H6;
- mismo sitio puede aportar varias tareas, pero seguirá siendo un único grupo para train/test.

Excluir una interfaz cuando su uso requiera acciones no autorizadas, datos personales reales, compra/pago o acceso a información privada.


## 6. Registro de tareas

In [6]:
task_registry_columns = [
    "task_id",
    "site_id",
    "task_name",
    "task_description",
    "start_condition",
    "success_criterion",
    "failure_criterion",
    "expected_page_type",
    "expected_interaction_type",
    "applicable_H1",
    "applicable_H4",
    "applicable_H5",
    "applicable_H6",
    "notes"
]

task_registry_template = pd.DataFrame(
    columns=task_registry_columns
)

display(
    task_registry_template
)


,task_id,site_id,task_name,task_description,start_condition,success_criterion,failure_criterion,expected_page_type,expected_interaction_type,applicable_H1,applicable_H4,applicable_H5,applicable_H6,notes


`applicable_H*` no es la etiqueta. Solo documenta si la tarea proporciona una oportunidad razonable para evaluar esa heurística. Si una heurística no puede observarse, la Y correspondiente debe permanecer `NaN`.


## 7. Registro de instancias y artefactos de evidencia

In [7]:
instance_registry_columns = [
    "instance_id",
    "site_id",
    "task_id",
    "execution_id",
    "collection_date",
    "browser",
    "viewport_width",
    "viewport_height",
    "extension_version",
    "task_completed",
    "extension_export_path",
    "screenshot_before_path",
    "screenshot_after_path",
    "dom_snapshot_before_path",
    "dom_snapshot_after_path",
    "interaction_trace_path",
    "recording_path",
    "collection_status",
    "notes"
]

instance_registry_template = pd.DataFrame(
    columns=instance_registry_columns
)

display(
    instance_registry_template
)


,instance_id,site_id,task_id,execution_id,collection_date,browser,viewport_width,viewport_height,extension_version,task_completed,extension_export_path,screenshot_before_path,screenshot_after_path,dom_snapshot_before_path,dom_snapshot_after_path,interaction_trace_path,recording_path,collection_status,notes


Los artefactos de evidencia para Y pueden incluir capturas, DOM/snapshot, secuencia de interacción o grabación. Al LLM **no** se le proporcionarán las métricas X agregadas, reglas de umbral ni predicciones del modelo.


## 8. Definición de las cuatro Y Silver

In [8]:
target_rubric = pd.DataFrame([
    {
        "target":
            "H1_violation",

        "heuristic":
            "Visibility of system status",

        "label_1":
            "Existe evidencia observable de que el sistema no comunica adecuadamente estado, progreso, resultado o feedback durante la interacción evaluada.",

        "label_0":
            "La tarea ofrece una oportunidad aplicable y no se observa una violación de H1 en la evidencia disponible.",

        "label_NA":
            "La evidencia/tarea no permite evaluar H1 con suficiente fundamento."
    },

    {
        "target":
            "H4_violation",

        "heuristic":
            "Consistency and standards",

        "label_1":
            "Existe inconsistencia observable de componentes, etiquetas, navegación o convenciones dentro del flujo evaluado.",

        "label_0":
            "Existe evidencia comparable suficiente y no se observa una violación de H4.",

        "label_NA":
            "No hay estados/páginas/componentes comparables suficientes para evaluar H4."
    },

    {
        "target":
            "H5_violation",

        "heuristic":
            "Error prevention",

        "label_1":
            "Existe una oportunidad real de error y la interfaz no lo previene adecuadamente mediante validación, bloqueo, confirmación o protección pertinente.",

        "label_0":
            "Existe una oportunidad aplicable de error y la interfaz implementa prevención adecuada en la evidencia observada.",

        "label_NA":
            "La tarea no expone una oportunidad de error relevante o la evidencia es insuficiente."
    },

    {
        "target":
            "H6_violation",

        "heuristic":
            "Recognition rather than recall",

        "label_1":
            "La interfaz obliga a recordar información/acciones que razonablemente deberían ser visibles o reconocibles, o presenta controles/opciones sin señalización suficiente.",

        "label_0":
            "La tarea permite evaluar reconocimiento y la información/opciones necesarias permanecen suficientemente visibles o identificables.",

        "label_NA":
            "La evidencia no permite evaluar H6 con suficiente fundamento."
    }
])

display(
    target_rubric
)


,target,heuristic,label_1,label_0,label_NA
0,H1_violation,Visibility of system status,"Existe evidencia observable de que el sistema no comunica adecuadamente estado, progreso, resultado o feedback durante la interacción evaluada.",La tarea ofrece una oportunidad aplicable y no se observa una violación de H1 en la evidencia disponible.,La evidencia/tarea no permite evaluar H1 con suficiente fundamento.
1,H4_violation,Consistency and standards,"Existe inconsistencia observable de componentes, etiquetas, navegación o convenciones dentro del flujo evaluado.",Existe evidencia comparable suficiente y no se observa una violación de H4.,No hay estados/páginas/componentes comparables suficientes para evaluar H4.
2,H5_violation,Error prevention,"Existe una oportunidad real de error y la interfaz no lo previene adecuadamente mediante validación, bloqueo, confirmación o protección pertinente.",Existe una oportunidad aplicable de error y la interfaz implementa prevención adecuada en la evidencia observada.,La tarea no expone una oportunidad de error relevante o la evidencia es insuficiente.
3,H6_violation,Recognition rather than recall,"La interfaz obliga a recordar información/acciones que razonablemente deberían ser visibles o reconocibles, o presenta controles/opciones sin señalización suficiente.",La tarea permite evaluar reconocimiento y la información/opciones necesarias permanecen suficientemente visibles o identificables.,La evidencia no permite evaluar H6 con suficiente fundamento.


## 9. Plantilla de evaluación LLM

Se recomiendan **tres ejecuciones de evaluación LLM por instancia**. Son repeticiones independientes de evaluación, no evaluadores humanos.

Cada ejecución debe registrar modelo, versión/fecha, `prompt_version`, `run_id`, respuesta binaria/NA, confianza y evidencia textual.


In [9]:
label_template_columns = [
    "instance_id",
    "label_run_id",
    "model_id",
    "model_version_or_date",
    "prompt_version",
    "H1_label",
    "H1_confidence",
    "H1_evidence",
    "H4_label",
    "H4_confidence",
    "H4_evidence",
    "H5_label",
    "H5_confidence",
    "H5_evidence",
    "H6_label",
    "H6_confidence",
    "H6_evidence",
    "annotation_timestamp",
    "annotation_status"
]

llm_label_template = pd.DataFrame(
    columns=label_template_columns
)

display(
    llm_label_template
)


,instance_id,label_run_id,model_id,model_version_or_date,prompt_version,H1_label,H1_confidence,H1_evidence,H4_label,H4_confidence,H4_evidence,H5_label,H5_confidence,H5_evidence,H6_label,H6_confidence,H6_evidence,annotation_timestamp,annotation_status


## 10. Prompt de anotación Silver

In [10]:
LLM_PROMPT_VERSION = "silver_v1"

llm_annotation_prompt = """# Nielsen Silver Annotation — silver_v1

You are evaluating one concrete web-interface task execution.

Evaluate ONLY the evidence supplied for this instance.
Do not infer behavior that is not observable.

Targets:
- H1: Visibility of system status
- H4: Consistency and standards
- H5: Error prevention
- H6: Recognition rather than recall

For each heuristic return:
- label: 1, 0, or null
- confidence: 0.0 to 1.0
- evidence: concise description of the observable evidence supporting the label

Rules:
1. label=1 only when an observable violation is supported.
2. label=0 only when the heuristic is applicable and sufficient evidence supports absence of a violation in the evaluated flow.
3. label=null when the heuristic is not assessable from the evidence.
4. Do not use absence of evidence as a negative label.
5. Do not invent pages, states, controls, errors, messages, delays, or user actions.
6. Judge the evaluated instance, not the general reputation or expected design of the site.
7. Do not use or request computed feature values, thresholds, model predictions, or prior labels.

Return JSON only:
{
  "H1": {"label": 0|1|null, "confidence": 0.0-1.0, "evidence": "..."},
  "H4": {"label": 0|1|null, "confidence": 0.0-1.0, "evidence": "..."},
  "H5": {"label": 0|1|null, "confidence": 0.0-1.0, "evidence": "..."},
  "H6": {"label": 0|1|null, "confidence": 0.0-1.0, "evidence": "..."}
}
"""

print(
    llm_annotation_prompt
)


# Nielsen Silver Annotation — silver_v1

You are evaluating one concrete web-interface task execution.

Evaluate ONLY the evidence supplied for this instance.
Do not infer behavior that is not observable.

Targets:
- H1: Visibility of system status
- H4: Consistency and standards
- H5: Error prevention
- H6: Recognition rather than recall

For each heuristic return:
- label: 1, 0, or null
- confidence: 0.0 to 1.0
- evidence: concise description of the observable evidence supporting the label

Rules:
1. label=1 only when an observable violation is supported.
2. label=0 only when the heuristic is applicable and sufficient evidence supports absence of a violation in the evaluated flow.
3. label=null when the heuristic is not assessable from the evidence.
4. Do not use absence of evidence as a negative label.
5. Do not invent pages, states, controls, errors, messages, delays, or user actions.
6. Judge the evaluated instance, not the general reputation or expected design of the site.
7. Do 

### Evidencia que puede recibir el LLM

Permitido:

- descripción de la tarea y criterio de éxito;
- screenshots antes/después;
- estados intermedios relevantes;
- DOM/snapshot bruto o extractos necesarios;
- registro bruto/secuencial de acciones y cambios observados;
- grabación de la ejecución.

No permitido para generar Y:

- `task_time_s`, `feedback_latency_ms`, ratios o demás X ya agregadas;
- umbrales usados por el clasificador;
- predicciones del clasificador;
- etiquetas obtenidas de reglas deterministas sobre las mismas X.


## 11. Política de consenso

In [11]:
label_consensus_policy = pd.DataFrame([
    {
        "rule":
            "RUNS_PER_INSTANCE",

        "value":
            3,

        "description":
            "Tres ejecuciones LLM independientes por instancia."
    },

    {
        "rule":
            "VALID_VALUES",

        "value":
            "0 | 1 | NaN",

        "description":
            "NaN significa no evaluable; nunca se convierte automáticamente en 0."
    },

    {
        "rule":
            "BINARY_MAJORITY",

        "value":
            "2_of_3",

        "description":
            "Si al menos dos ejecuciones válidas coinciden en 0 o 1, se acepta el consenso."
    },

    {
        "rule":
            "INSUFFICIENT_VALID_RUNS",

        "value":
            "NaN",

        "description":
            "Con menos de dos etiquetas binarias válidas no se asigna target."
    },

    {
        "rule":
            "CONFLICT",

        "value":
            "REVIEW_OR_NA",

        "description":
            "Conflictos sin mayoría válida permanecen NaN o se revisan en análisis separado."
    },

    {
        "rule":
            "FINAL_LABEL_LEVEL",

        "value":
            "LLM_DIRECT_SILVER",

        "description":
            "Procedencia Silver; no gold/expert."
    }
])

display(
    label_consensus_policy
)


,rule,value,description
0,RUNS_PER_INSTANCE,3,Tres ejecuciones LLM independientes por instancia.
1,VALID_VALUES,0 | 1 | NaN,NaN significa no evaluable; nunca se convierte automáticamente en 0.
2,BINARY_MAJORITY,2_of_3,"Si al menos dos ejecuciones válidas coinciden en 0 o 1, se acepta el consenso."
3,INSUFFICIENT_VALID_RUNS,NaN,Con menos de dos etiquetas binarias válidas no se asigna target.
4,CONFLICT,REVIEW_OR_NA,Conflictos sin mayoría válida permanecen NaN o se revisan en análisis separado.
5,FINAL_LABEL_LEVEL,LLM_DIRECT_SILVER,Procedencia Silver; no gold/expert.


## 12. Función reproducible de consenso

In [12]:
def majority_silver(
    values
):

    valid = []

    for value in values:

        if pd.isna(
            value
        ):
            continue

        try:
            value = int(
                value
            )
        except Exception:
            continue

        if value in {
            0,
            1
        }:
            valid.append(
                value
            )

    if len(
        valid
    ) < 2:
        return np.nan

    n0 = sum(
        value == 0
        for value
        in valid
    )

    n1 = sum(
        value == 1
        for value
        in valid
    )

    if n1 >= 2:
        return 1

    if n0 >= 2:
        return 0

    return np.nan


assert majority_silver(
    [1, 1, 0]
) == 1

assert majority_silver(
    [0, 0, np.nan]
) == 0

assert pd.isna(
    majority_silver(
        [1, np.nan, np.nan]
    )
)

print(
    "Función de consenso ✅"
)


Función de consenso ✅


## 13. Criterios provisionales de tamaño y balance

In [13]:
collection_targets = pd.DataFrame([
    {
        "phase":
            "PILOT",

        "min_sites":
            12,

        "min_instances":
            48,

        "min_per_class_per_target":
            None,

        "purpose":
            "Validar captura, esquema, anotación y observabilidad; no entrenar modelo final."
    },

    {
        "phase":
            "DEVELOPMENT",

        "min_sites":
            30,

        "min_instances":
            200,

        "min_per_class_per_target":
            40,

        "purpose":
            "Criterio operativo provisional para comenzar modelado serio; la suficiencia final se comprobará con grouped CV y learning curves."
    }
])

display(
    collection_targets
)


,phase,min_sites,min_instances,min_per_class_per_target,purpose
0,PILOT,12,48,NaN,"Validar captura, esquema, anotación y observabilidad; no entrenar modelo final."
1,DEVELOPMENT,30,200,40.0,Criterio operativo provisional para comenzar modelado serio; la suficiencia final se comprobará con grouped CV y learning curves.


Los valores anteriores son **criterios operativos del proyecto**, no umbrales universales ni requisitos de Nielsen/ISO. Si una heurística continúa muy desbalanceada, se amplía la recolección antes de entrenarla.


## 14. Auditoría de balance antes de ML

In [14]:
def target_balance_table(
    df,
    target_cols=None
):

    if target_cols is None:

        target_cols = [
            "H1_violation",
            "H4_violation",
            "H5_violation",
            "H6_violation"
        ]

    records = []

    for col in target_cols:

        if col not in df.columns:

            records.append({
                "target":
                    col,

                "n":
                    0,

                "n_0":
                    0,

                "n_1":
                    0,

                "n_na":
                    len(
                        df
                    ),

                "both_classes":
                    False
            })

            continue

        values = pd.to_numeric(
            df[
                col
            ],
            errors="coerce"
        )

        n0 = int(
            (
                values
                ==
                0
            ).sum()
        )

        n1 = int(
            (
                values
                ==
                1
            ).sum()
        )

        nna = int(
            values
            .isna()
            .sum()
        )

        records.append({
            "target":
                col,

            "n":
                int(
                    values
                    .notna()
                    .sum()
                ),

            "n_0":
                n0,

            "n_1":
                n1,

            "n_na":
                nna,

            "both_classes":
                (
                    n0 > 0
                    and
                    n1 > 0
                )
        })

    return pd.DataFrame(
        records
    )


## 15. Política de split

In [15]:
split_policy = pd.DataFrame([
    {
        "item":
            "GROUP_KEY",

        "value":
            "site_id",

        "reason":
            "Todas las tareas/ejecuciones del mismo sitio permanecen en un único split."
    },

    {
        "item":
            "ROW_RANDOM_SPLIT",

        "value":
            False,

        "reason":
            "Evitar fuga de identidad visual/estructural del mismo sitio."
    },

    {
        "item":
            "PRIMARY_EVALUATION",

        "value":
            "GROUPED_CV",

        "reason":
            "Evaluación por grupos de sitios."
    },

    {
        "item":
            "OPTIONAL_ROBUSTNESS",

        "value":
            "LEAVE_ONE_SITE_OR_SOURCE_OUT",

        "reason":
            "Medir generalización a sitios no vistos cuando el número de grupos lo permita."
    }
])

display(
    split_policy
)


,item,value,reason
0,GROUP_KEY,site_id,Todas las tareas/ejecuciones del mismo sitio permanecen en un único split.
1,ROW_RANDOM_SPLIT,False,Evitar fuga de identidad visual/estructural del mismo sitio.
2,PRIMARY_EVALUATION,GROUPED_CV,Evaluación por grupos de sitios.
3,OPTIONAL_ROBUSTNESS,LEAVE_ONE_SITE_OR_SOURCE_OUT,Medir generalización a sitios no vistos cuando el número de grupos lo permita.


## 16. Validaciones del contrato

In [16]:
assert (
    extension_capture_contract[
        "column"
    ]
    .duplicated()
    .sum()
    ==
    0
)

assert (
    (
        extension_capture_contract[
            "role"
        ]
        ==
        "X"
    ).sum()
    ==
    40
)

assert (
    len(
        set(
            X_NAMES
        )
    )
    ==
    40
)

assert (
    set(
        [
            "H1_violation",
            "H4_violation",
            "H5_violation",
            "H6_violation"
        ]
    )
    .isdisjoint(
        set(
            X_NAMES
        )
    )
)

print(
    "Contrato X/Y validado ✅"
)


Contrato X/Y validado ✅


## 17. Guardar contratos y plantillas

In [17]:
extension_capture_contract.to_csv(
    REPORTS_DIR
    / "silver_extension_capture_contract.csv",
    index=False
)

observation_unit_policy.to_csv(
    REPORTS_DIR
    / "silver_observation_unit_policy.csv",
    index=False
)

target_rubric.to_csv(
    REPORTS_DIR
    / "silver_target_rubric.csv",
    index=False
)

label_consensus_policy.to_csv(
    REPORTS_DIR
    / "silver_label_consensus_policy.csv",
    index=False
)

collection_targets.to_csv(
    REPORTS_DIR
    / "silver_collection_targets.csv",
    index=False
)

split_policy.to_csv(
    REPORTS_DIR
    / "silver_split_policy.csv",
    index=False
)

site_registry_template.to_csv(
    SILVER_METADATA_DIR
    / "site_registry.csv",
    index=False
)

task_registry_template.to_csv(
    SILVER_METADATA_DIR
    / "task_registry.csv",
    index=False
)

instance_registry_template.to_csv(
    SILVER_METADATA_DIR
    / "instance_registry.csv",
    index=False
)

llm_label_template.to_csv(
    SILVER_LABEL_DIR
    / "llm_labels_long.csv",
    index=False
)

prompt_path = (
    REPORTS_DIR
    / "silver_llm_annotation_prompt.md"
)

prompt_path.write_text(
    llm_annotation_prompt,
    encoding="utf-8"
)


1398

In [18]:
expected = [
    REPORTS_DIR
    / "silver_extension_capture_contract.csv",

    REPORTS_DIR
    / "silver_observation_unit_policy.csv",

    REPORTS_DIR
    / "silver_target_rubric.csv",

    REPORTS_DIR
    / "silver_label_consensus_policy.csv",

    REPORTS_DIR
    / "silver_collection_targets.csv",

    REPORTS_DIR
    / "silver_split_policy.csv",

    REPORTS_DIR
    / "silver_llm_annotation_prompt.md",

    SILVER_METADATA_DIR
    / "site_registry.csv",

    SILVER_METADATA_DIR
    / "task_registry.csv",

    SILVER_METADATA_DIR
    / "instance_registry.csv",

    SILVER_LABEL_DIR
    / "llm_labels_long.csv"
]

for path in expected:

    print(
        path.relative_to(
            PROJECT_DIR
        ),
        "✅"
        if path.exists()
        else "❌"
    )


reports/tables/silver_extension_capture_contract.csv ✅
reports/tables/silver_observation_unit_policy.csv ✅
reports/tables/silver_target_rubric.csv ✅
reports/tables/silver_label_consensus_policy.csv ✅
reports/tables/silver_collection_targets.csv ✅
reports/tables/silver_split_policy.csv ✅
reports/tables/silver_llm_annotation_prompt.md ✅
data/silver/metadata/site_registry.csv ✅
data/silver/metadata/task_registry.csv ✅
data/silver/metadata/instance_registry.csv ✅
data/silver/labels/llm_labels_long.csv ✅


## Resultado esperado

Después de ejecutar este notebook deberá existir:

```text
data/silver/
├── metadata/
│   ├── site_registry.csv
│   ├── task_registry.csv
│   └── instance_registry.csv
├── raw/
├── labels/
│   └── llm_labels_long.csv
└── interim/
```

La carpeta `data/processed/` permanece sin modificarse.

## Siguiente paso

El notebook 25 debe convertir el contrato anterior en un **piloto de recolección**:

1. seleccionar los primeros sitios educativos;
2. definir 1–3 tareas reproducibles por sitio;
3. capturar X mediante la extensión;
4. guardar los artefactos de evidencia;
5. ejecutar las tres anotaciones LLM;
6. construir consenso;
7. auditar balance H1/H4/H5/H6.

No se entrenará Random Forest hasta que el piloto valide el pipeline completo y cada target tenga ambas clases observadas.
